In [3]:
import os
# os.environ["KERAS_BACKEND"] = "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import zipfile
from pathlib import Path
import pickle
import tarfile
import datetime
import numpy as np
import urllib.request
import sklearn.metrics
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import tensorboard as tb
import albumentations as A

import cv2

N_CLASSES = 200

2026-05-31 21:12:23.075351: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/foler/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:993: The handshake operation timed out>
  data = fetch_version_info()


In [4]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    print(gpus)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
# import torch
# print(keras.backend.backend())
# print(f"CUDA available: {torch.cuda.is_available()}")
# if torch.cuda.is_available():
#     print(f"GPU name: {torch.cuda.get_device_name(0)}")
#     print(f"Current device index: {torch.cuda.current_device()}")

In [6]:
BATCH_SIZE = 128
HISTORY_DIR = Path('./history')
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR_ROOT = Path('./dataset')
DATASET_DIR_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_DIR = DATASET_DIR_ROOT / 'tiny-imagenet-200'

In [7]:
def download_data():
    dataset = DATASET_DIR
    dataset_zip = dataset.with_suffix('.zip')

    if not dataset_zip.exists():
        urllib.request.urlretrieve('http://cs231n.stanford.edu/tiny-imagenet-200.zip', dataset_zip)

    if not dataset.exists():
        file = zipfile.ZipFile(dataset_zip, 'r')
        file.extractall(path=DATASET_DIR_ROOT)

In [8]:
download_data()

In [9]:
with open(DATASET_DIR / "wnids.txt", "r") as fc:
    classes = [l.strip() for l in fc.readlines() ]
classes_map = { c: i for i, c in enumerate(classes) }
classes_map

{'n02124075': 0,
 'n04067472': 1,
 'n04540053': 2,
 'n04099969': 3,
 'n07749582': 4,
 'n01641577': 5,
 'n02802426': 6,
 'n09246464': 7,
 'n07920052': 8,
 'n03970156': 9,
 'n03891332': 10,
 'n02106662': 11,
 'n03201208': 12,
 'n02279972': 13,
 'n02132136': 14,
 'n04146614': 15,
 'n07873807': 16,
 'n02364673': 17,
 'n04507155': 18,
 'n03854065': 19,
 'n03838899': 20,
 'n03733131': 21,
 'n01443537': 22,
 'n07875152': 23,
 'n03544143': 24,
 'n09428293': 25,
 'n03085013': 26,
 'n02437312': 27,
 'n07614500': 28,
 'n03804744': 29,
 'n04265275': 30,
 'n02963159': 31,
 'n02486410': 32,
 'n01944390': 33,
 'n09256479': 34,
 'n02058221': 35,
 'n04275548': 36,
 'n02321529': 37,
 'n02769748': 38,
 'n02099712': 39,
 'n07695742': 40,
 'n02056570': 41,
 'n02281406': 42,
 'n01774750': 43,
 'n02509815': 44,
 'n03983396': 45,
 'n07753592': 46,
 'n04254777': 47,
 'n02233338': 48,
 'n04008634': 49,
 'n02823428': 50,
 'n02236044': 51,
 'n03393912': 52,
 'n07583066': 53,
 'n04074963': 54,
 'n01629819': 55,
 '

In [10]:
def get_dataset_part(path_to_dir_with_annotations: Path, is_val: bool = False) -> tuple[list[Path], list[int]]:
    _path = path_to_dir_with_annotations
    assert _path.is_dir()

    metadata_path = list(_path.glob("*.txt"))[0]

    with open(metadata_path, "r") as fm:
        _md = np.loadtxt(fm, dtype=str)
        file_paths = [_path / "images" / name for name in _md[:, 0]]
        if is_val:
            name_classes = list(map(str, _md[:, 1]))
        else:
            name_classes = [_path.name] * len(file_paths)

    i_classes = [classes_map[n] for n in name_classes]

    return file_paths, i_classes

def get_train_dataset(path_to_dir_with_train_dirs: Path) -> tuple[list[Path], list[int]]:
    _path = path_to_dir_with_train_dirs
    assert _path.is_dir()

    file_paths = []
    i_classes = []
    for class_dir in _path.iterdir():
        fps, cid = get_dataset_part(class_dir)
        file_paths.extend(fps)
        i_classes.extend(cid)

    return file_paths, i_classes

In [11]:
# get_dataset_part(DATASET_DIR / "val", is_val=True)
# get_dataset_part(DATASET_DIR / "train" / "n01443537")

# td = get_train_dataset(DATASET_DIR / "train")

In [12]:
# list(map(len, td))

In [13]:
class Dataset(keras.utils.PyDataset):

    def __init__(
            self, 
            image_paths: list[Path], 
            image_classes: list[int],
            batch_size: int,
            seed: int | None = None, 
            shuffle: bool = False, 
            augmentation: A.Compose | None = None,
            **kwargs):
        super().__init__(**kwargs)
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.augmentation = augmentation

        self.image_paths = image_paths
        self.image_classes = np.array(image_classes)

        self.image_index = np.arange(len(image_paths))

        self.on_epoch_end()

    def make_train_dataset(batch_size: int, seed: int | None = None, shuffle: bool = False, **kwargs):
        aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(scale=(0.9, 1.1), rotate=10, translate_percent=0.05, p=0.4),
            A.RandomBrightnessContrast(p=0.2),
        ], p=1.0)

        return Dataset(*get_train_dataset(DATASET_DIR / "train"), batch_size, seed, shuffle, augmentation=aug, **kwargs)
    
    def make_validation_dataset(batch_size: int, seed: int | None = None, shuffle: bool = False, **kwargs):
        return Dataset(*get_dataset_part(DATASET_DIR / "val", is_val=True), batch_size, seed, shuffle, **kwargs)

    def __len__(self):
        return (len(self.image_paths) + self.batch_size - 1) // self.batch_size

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.image_index)
    
    def __getitem__(self, index: int):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch = self.image_index[start:end]

        image_f = lambda x: x
        if self.augmentation:
            image_f = lambda x: self.augmentation(image=x)["image"]

        images = np.array([image_f(cv2.imread(self.image_paths[idx])) for idx in batch])
        return images, self.image_classes[batch]


In [14]:
train_dataset = Dataset.make_train_dataset(batch_size=BATCH_SIZE, seed=42, shuffle=True)

In [15]:
val_dataset = Dataset.make_validation_dataset(batch_size=BATCH_SIZE, seed=42, shuffle=False)

In [16]:
def res_block(inp, dim: int, name: str | None = None, is_first: bool=False):
    x = inp

    half_text = "_strides2" if is_first else ""

    x = keras.layers.Conv2D(dim // 4, 1, strides=1 + is_first, padding='same', name=f"{name}_conv_1{half_text}")(x)
    x = keras.layers.BatchNormalization(name=f"{name}_bn_1")(x)
    x = keras.layers.ReLU(name=f"{name}_swish_1")(x)

    x = keras.layers.Conv2D(dim // 4, 3, padding='same', name=f"{name}_conv_2")(x)
    x = keras.layers.BatchNormalization(name=f"{name}_bn_2")(x)
    x = keras.layers.ReLU(name=f"{name}_swish_2")(x)

    x = keras.layers.Conv2D(dim, 1, name=f"{name}_conv_3")(x)
    if (not is_first):
        x = keras.layers.Add(name=f"{name}_add")([x, inp])

    x = keras.layers.ReLU(name=f"{name}_relu_3")(x)
    x = keras.layers.Dropout(0.1, name=f"{name}_dropout")(x)

    return x

In [17]:
SEED = 42

FILTERS = 64 

x = inputs = keras.layers.Input((64, 64, 3), name='inp')

x = keras.layers.Rescaling(1./255)(x)

x = keras.layers.Conv2D(FILTERS, 7, strides=2, padding='same', name="inp_conv")(x)
x = keras.layers.BatchNormalization(name="bn_1")(x)
# x = keras.layers.MaxPool2D(3, strides=2, padding='same', name="inp_max")(x)

# nums = [
#     (64, 3),
#     (128, 4),
#     (256, 6),
#     (512, 3)
# ]

# nums = [
#     (256, 2),
#     (512, 2),
#     (1024, 2),
#     (2048, 2)
# ]

nums = [
    (FILTERS * 4, 1),
    (FILTERS * 8, 1),
    (FILTERS * 16, 1),
    # (FILTERS * 32, 2)
]

for i, (ker, times) in enumerate(nums):
    for j in range(times):
        # isf = (j == 0 and i > 0)
        isf = j==0
        x = res_block(x, ker, name=f"conv_{i}_{j}", is_first=isf)

x = keras.layers.GlobalAveragePooling2D(name="avg_pool_end")(x)

x = keras.layers.Dropout(0.3, seed=SEED, name="dropout_1")(x)
x = keras.layers.Dense(N_CLASSES, activation='softmax', name="dense_softmax")(x)

model = keras.models.Model(inputs, x)

I0000 00:00:1780225957.722526   22551 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9709 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [18]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inp (InputLayer)                │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inp_conv (Conv2D)               │ (None, 32, 32, 64)     │         9,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn_1 (BatchNormalization)       │ (None, 32, 32, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_conv_1_strides2        │ (None, 16, 16, 64)     │         4,160 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_bn_1                   │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_swish_1 (ReLU)         │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_conv_2 (Conv2D)        │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_bn_2                   │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_swish_2 (ReLU)         │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_conv_3 (Conv2D)        │ (None, 16, 16, 256)    │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_relu_3 (ReLU)          │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_0_0_dropout (Dropout)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_conv_1_strides2        │ (None, 8, 8, 128)      │        32,896 │
│ (Conv2D)                        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_bn_1                   │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_swish_1 (ReLU)         │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_conv_2 (Conv2D)        │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_bn_2                   │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_swish_2 (ReLU)         │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_conv_3 (Conv2D)        │ (None, 8, 8, 512)      │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_relu_3 (ReLU)          │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1_0_dropout (Dropout)      │ (None, 8, 8, 512)      │             

 Total params: 1,507,144 (5.75 MB)

 Trainable params: 1,505,224 (5.74 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [19]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(), 
    metrics=[
        keras.metrics.SparseCategoricalAccuracy()
    ]
)

In [20]:
# logdir = HISTORY_DIR / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir = HISTORY_DIR / "20260531-202627"
logdir.mkdir(parents=True, exist_ok=True)
logdir

PosixPath('history/20260531-202627')

In [21]:
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    logdir / 'model.keras',
    save_best_only=True
)

In [22]:
tb_log = logdir / 'logs'
tb_log
tensorboard_callback = keras.callbacks.TensorBoard(
    tb_log,
)
tb_log_str = str(tb_log).replace("\\", "/")
tb_log_str

'history/20260531-202627/logs'

In [23]:
%load_ext tensorboard

In [24]:
tb.notebook.list()

Known TensorBoard instances:
  - port 6007: logdir history/20260531-202627/logs (started 0:46:22 ago; pid 16683)
  - port 6006: logdir history/20260531-195226/logs (started 1:20:23 ago; pid 12660)


In [25]:
%tensorboard --logdir {tb_log_str}

Reusing TensorBoard on port 6007 (pid 16683), started 0:46:24 ago. (Use '!kill 16683' to kill it.)

In [42]:
# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [43]:
# import gc
# tf.keras.backend.clear_session()
# gc.collect()

In [26]:
# model.load_weights(model_checkpoint_callback.filepath)

/home/foler/miniconda3/envs/tf_gpu/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 74 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
# model.save(logdir / "model_hand_saved.keras")

In [100]:
model.fit(
    train_dataset, 
    validation_data=val_dataset, 
    batch_size=BATCH_SIZE,
    epochs=100,
    callbacks=[
        model_checkpoint_callback, 
        tensorboard_callback,
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-8),
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    ]
)

Epoch 1/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 73ms/step - loss: 4.7157 - sparse_categorical_accuracy: 0.0500 - val_loss: 4.6527 - val_sparse_categorical_accuracy: 0.0550 - learning_rate: 0.0010
Epoch 2/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 4.1171 - sparse_categorical_accuracy: 0.1136 - val_loss: 4.2151 - val_sparse_categorical_accuracy: 0.1125 - learning_rate: 0.0010
Epoch 3/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 3.8091 - sparse_categorical_accuracy: 0.1567 - val_loss: 3.9903 - val_sparse_categorical_accuracy: 0.1387 - learning_rate: 0.0010
Epoch 4/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 3.6255 - sparse_categorical_accuracy: 0.1874 - val_loss: 3.8042 - val_sparse_categorical_accuracy: 0.1660 - learning_rate: 0.0010
Epoch 5/100
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 24ms/step - loss: 3.4893 - sparse_categorical_accuracy: 0.2082 - val_loss: 3.7489 - val_sparse_categorical_accuracy: 0.1797 - learning_rate: 0.0010
Epoch 6/100
782/782 ━━━━━━━━━━━━━━━

In [28]:
y_true = val_dataset.image_classes
y_pred = np.argmax(model.predict(val_dataset), axis=1)

79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [29]:
sklearn.metrics.accuracy_score(y_true, y_pred)

0.4298

In [103]:
_, ax = plt.subplots(figsize=(75, 75))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)

plt.tight_layout()
plt.savefig(logdir / 'valid.png')

In [30]:
_, ax = plt.subplots(figsize=(75, 75))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False, normalize='true')

plt.tight_layout()
plt.savefig(logdir / 'valid_true.png')